In [2]:
import os
import cv2
import random

# =========================
# USER INPUT PATHS
# =========================

source_path = r"/mnt/lustre/lalit47/us_data/bench/LUMINOUS_Database"
target_path = r"/mnt/lustre/lalit47/down/seg/LUMINOUS"

# Source folders
images_folder = os.path.join(source_path, "B-mode")
masks_folder = os.path.join(source_path, "Masks")

# Target folders
train_images_out = os.path.join(target_path, "train", "images")
train_masks_out = os.path.join(target_path, "train", "masks")

val_images_out = os.path.join(target_path, "val", "images")
val_masks_out = os.path.join(target_path, "val", "masks")

test_images_out = os.path.join(target_path, "test", "images")
test_masks_out = os.path.join(target_path, "test", "masks")

# Create target directories
for folder in [
    train_images_out, train_masks_out,
    val_images_out, val_masks_out,
    test_images_out, test_masks_out
]:
    os.makedirs(folder, exist_ok=True)


def process_and_save(src, dst, is_mask=False):
    """
    Read image in grayscale, resize to 256x256, save as PNG.
    For masks, nearest-neighbor interpolation preserves label values.
    """

    im = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if im is None:
        print(f"Failed to read: {src}")
        return False

    interpolation = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    im = cv2.resize(im, (256, 256), interpolation=interpolation)

    cv2.imwrite(dst, im)
    return True


def get_case_names():
    """
    Find all paired image-mask cases.

    Expected:
        B-mode/1_1_Bmode.tif
        Masks/1_1_Mask.tif
    """

    case_names = []

    for file in os.listdir(images_folder):
        if file.endswith("_Bmode.tif"):
            name = file.replace("_Bmode.tif", "")

            image_path = os.path.join(images_folder, name + "_Bmode.tif")
            mask_path = os.path.join(masks_folder, name + "_Mask.tif")

            if os.path.exists(image_path) and os.path.exists(mask_path):
                case_names.append(name)
            else:
                print(f"Skipping {name}: matching mask not found")

    return sorted(case_names)


def split_7_1_2(case_names, seed=42):
    """
    Split data into train:val:test = 7:1:2
    without using sklearn.
    """

    case_names = case_names.copy()
    random.seed(seed)
    random.shuffle(case_names)

    total = len(case_names)

    train_end = int(total * 0.7)
    val_end = int(total * 0.8)

    train_cases = case_names[:train_end]
    val_cases = case_names[train_end:val_end]
    test_cases = case_names[val_end:]

    return train_cases, val_cases, test_cases


def copy_dataset(case_names, img_output, mask_output):
    """
    Process images and masks for a given split.
    """

    for name in case_names:
        image_src = os.path.join(images_folder, name + "_Bmode.tif")
        mask_src = os.path.join(masks_folder, name + "_Mask.tif")

        image_dst = os.path.join(img_output, name + ".png")
        mask_dst = os.path.join(mask_output, name + ".png")

        process_and_save(image_src, image_dst, is_mask=False)
        process_and_save(mask_src, mask_dst, is_mask=True)


# =========================
# Create 7:1:2 split
# =========================

all_cases = get_case_names()

train_cases, val_cases, test_cases = split_7_1_2(all_cases, seed=42)

print(f"Total cases: {len(all_cases)}")
print(f"Train cases: {len(train_cases)}")
print(f"Val cases:   {len(val_cases)}")
print(f"Test cases:  {len(test_cases)}")

# =========================
# Process datasets
# =========================

copy_dataset(train_cases, train_images_out, train_masks_out)
copy_dataset(val_cases, val_images_out, val_masks_out)
copy_dataset(test_cases, test_images_out, test_masks_out)

print("\nDataset preprocessing completed successfully!")

Skipping 100_1: matching mask not found
Skipping 95_30: matching mask not found
Skipping 22_22: matching mask not found
Skipping 61_32: matching mask not found
Skipping 46_1: matching mask not found
Skipping 27_1: matching mask not found
Skipping 102_1: matching mask not found
Skipping 71_24: matching mask not found
Skipping 4_1: matching mask not found
Skipping 22_5: matching mask not found
Skipping 5_1: matching mask not found
Skipping 20_1: matching mask not found
Skipping 78_22: matching mask not found
Skipping 4_29: matching mask not found
Skipping 68_1: matching mask not found
Skipping 59_2: matching mask not found
Skipping 24_20: matching mask not found
Skipping 3_22: matching mask not found
Skipping 61_1: matching mask not found
Skipping 73_1: matching mask not found
Skipping 103_1: matching mask not found
Skipping 33_22: matching mask not found
Skipping 78_1: matching mask not found
Skipping 107_1: matching mask not found
Skipping 33_1: matching mask not found
Skipping 62_1: m

[ WARN:0@61.272] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@61.273] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc697) encountered
[ WARN:0@63.113] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@63.113] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc697) encountered
[ WARN:0@67.334] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@67.334] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc697) encountered
[ WARN:0@67.859] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@67.859] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc697) encountered



Dataset preprocessing completed successfully!


In [3]:
import os
import cv2
import random
import numpy as np

# =========================
# USER INPUT PATHS
# =========================

source_path = r"/mnt/lustre/lalit47/us_data/bench/LUMINOUS_Database"
target_path = r"/mnt/lustre/lalit47/down/seg/LUMINOUS"

# Source folders
images_folder = os.path.join(source_path, "B-mode")
masks_folder = os.path.join(source_path, "Masks")

# Target folders
train_images_out = os.path.join(target_path, "train", "images")
train_masks_out = os.path.join(target_path, "train", "masks")

val_images_out = os.path.join(target_path, "val", "images")
val_masks_out = os.path.join(target_path, "val", "masks")

test_images_out = os.path.join(target_path, "test", "images")
test_masks_out = os.path.join(target_path, "test", "masks")

# Create target directories
for folder in [
    train_images_out, train_masks_out,
    val_images_out, val_masks_out,
    test_images_out, test_masks_out
]:
    os.makedirs(folder, exist_ok=True)


def process_and_save_image(src, dst):
    """
    Read B-mode image in grayscale,
    resize to 256x256,
    save as PNG.
    """

    im = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if im is None:
        print(f"Failed to read image: {src}")
        return False

    im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_LINEAR)
    cv2.imwrite(dst, im)

    return True


def find_mask_files(name):
    """
    Find all mask files for one case.

    Examples:
        1_1_Mask.tif
        1_1_Mask1.tif
        1_1_Mask2.tif
    """

    mask_files = []

    for file in os.listdir(masks_folder):
        if (
            file.startswith(name + "_Mask")
            and file.endswith(".tif")
        ):
            mask_files.append(os.path.join(masks_folder, file))

    return sorted(mask_files)


def combine_and_save_masks(mask_files, dst):
    """
    Combine multiple mask files into a single binary mask.

    Any non-zero pixel in any mask becomes foreground.
    """

    combined_mask = None

    for mask_file in mask_files:
        mask = cv2.imread(mask_file, cv2.IMREAD_GRAYSCALE)

        if mask is None:
            print(f"Failed to read mask: {mask_file}")
            continue

        # Convert to binary mask: background=0, foreground=255
        mask = (mask > 0).astype(np.uint8) * 255

        if combined_mask is None:
            combined_mask = mask
        else:
            combined_mask = np.maximum(combined_mask, mask)

    if combined_mask is None:
        print(f"No valid masks found for: {dst}")
        return False

    combined_mask = cv2.resize(
        combined_mask,
        (256, 256),
        interpolation=cv2.INTER_NEAREST
    )

    cv2.imwrite(dst, combined_mask)

    return True


def get_case_names():
    """
    Find all paired image-mask cases.

    Expected image:
        B-mode/1_1_Bmode.tif

    Expected mask examples:
        Masks/1_1_Mask.tif
        Masks/1_1_Mask1.tif
        Masks/1_1_Mask2.tif
    """

    case_names = []

    for file in os.listdir(images_folder):
        if file.endswith("_Bmode.tif"):
            name = file.replace("_Bmode.tif", "")

            image_path = os.path.join(images_folder, name + "_Bmode.tif")
            mask_files = find_mask_files(name)

            if os.path.exists(image_path) and len(mask_files) > 0:
                case_names.append(name)
            else:
                print(f"Skipping {name}: matching mask not found")

    return sorted(case_names)


def split_7_1_2(case_names, seed=42):
    """
    Split data into train:val:test = 7:1:2
    without using sklearn.
    """

    case_names = case_names.copy()
    random.seed(seed)
    random.shuffle(case_names)

    total = len(case_names)

    train_end = int(total * 0.7)
    val_end = int(total * 0.8)

    train_cases = case_names[:train_end]
    val_cases = case_names[train_end:val_end]
    test_cases = case_names[val_end:]

    return train_cases, val_cases, test_cases


def copy_dataset(case_names, img_output, mask_output):
    """
    Process images and combined masks for a given split.
    """

    for name in case_names:
        image_src = os.path.join(images_folder, name + "_Bmode.tif")
        mask_files = find_mask_files(name)

        image_dst = os.path.join(img_output, name + ".png")
        mask_dst = os.path.join(mask_output, name + ".png")

        if os.path.exists(image_src):
            process_and_save_image(image_src, image_dst)
        else:
            print(f"Image Missing: {image_src}")

        if len(mask_files) > 0:
            combine_and_save_masks(mask_files, mask_dst)
        else:
            print(f"Mask Missing for: {name}")


# =========================
# Create 7:1:2 split
# =========================

all_cases = get_case_names()

train_cases, val_cases, test_cases = split_7_1_2(all_cases, seed=42)

print(f"Total cases: {len(all_cases)}")
print(f"Train cases: {len(train_cases)}")
print(f"Val cases:   {len(val_cases)}")
print(f"Test cases:  {len(test_cases)}")

# =========================
# Process datasets
# =========================

copy_dataset(train_cases, train_images_out, train_masks_out)
copy_dataset(val_cases, val_images_out, val_masks_out)
copy_dataset(test_cases, test_images_out, test_masks_out)

print("\nDataset preprocessing completed successfully!")

Total cases: 341
Train cases: 238
Val cases:   34
Test cases:  69


[ WARN:0@598.765] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@598.765] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc697) encountered
[ WARN:0@598.789] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@598.789] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc697) encountered
[ WARN:0@598.809] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@598.809] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc697) encountered
[ WARN:0@598.828] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@598.828] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc697) enco


Dataset preprocessing completed successfully!


[ WARN:0@608.236] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@608.236] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc697) encountered
[ WARN:0@608.245] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@608.245] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc697) encountered
